# Class Imbalance

When one class is rare, a model can achieve high accuracy by mostly predicting the majority class while missing the examples of interest. This notebook compares class weighting with random oversampling and SMOTE. Fit and resample using training data only, then compare methods on the same held-out test set.


## 1. Create and split an imbalanced dataset

Here class `1` is the minority class. The synthetic data makes it easy to illustrate imbalance; in practice, check class counts and define which class is important before choosing metrics or a balancing method.


In [1]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split


X, y = make_classification(
    n_samples=1200,
    n_features=6,
    n_informative=4,
    n_redundant=0,
    weights=[0.95, 0.05],
    random_state=42,
)

class_proportions = pd.Series(y).value_counts(normalize=True).sort_index()
class_proportions.index.name = "class"
class_proportions.name = "proportion"
class_proportions


class
0    0.945
1    0.055
Name: proportion, dtype: float64

In [2]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)


## 2. Establish an unbalanced baseline

First compare a model that treats both classes equally with a trivial model that always predicts the majority class. The trivial model can have high accuracy and zero recall for class 1. Use the same held-out test set for each comparison.


In [3]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler


baseline = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000),
)
baseline.fit(X_train, y_train)
baseline_predictions = baseline.predict(X_test)

majority_predictions = np.zeros_like(y_test)
print(
    "Always-majority accuracy:",
    round(accuracy_score(y_test, majority_predictions), 3),
)
print(
    "Always-majority recall for class 1:",
    round(recall_score(y_test, majority_predictions, zero_division=0), 3),
)
print(
    "Baseline logistic-regression metrics:",
    {
        "accuracy": round(accuracy_score(y_test, baseline_predictions), 3),
        "minority precision": round(
            precision_score(y_test, baseline_predictions, zero_division=0), 3
        ),
        "minority recall": round(
            recall_score(y_test, baseline_predictions, zero_division=0), 3
        ),
        "minority F1": round(
            f1_score(y_test, baseline_predictions, zero_division=0), 3
        ),
    },
)


Always-majority accuracy: 0.946
Always-majority recall for class 1: 0.0
Baseline logistic-regression metrics: {'accuracy': 0.958, 'minority precision': 1.0, 'minority recall': 0.231, 'minority F1': 0.375}


## 3. Weight minority-class errors more heavily

`class_weight="balanced"` increases the influence of the less frequent class during fitting, without changing the number of training rows. Compare it with the unweighted baseline on the same test data. It may improve minority recall, but can also increase false positives; it does not guarantee better overall performance.

Scikit-learn sets each class weight inversely to its frequency: $n_{samples}/(n_{classes}\times n_{class})$. This is a simple first approach because it needs no resampling.


In [4]:
balanced_model = make_pipeline(
    StandardScaler(),
    LogisticRegression(class_weight="balanced", max_iter=1000),
)
balanced_model.fit(X_train, y_train)
balanced_predictions = balanced_model.predict(X_test)

pd.DataFrame(
    [
        {
            "model": "unweighted",
            "accuracy": accuracy_score(y_test, baseline_predictions),
            "minority precision": precision_score(
                y_test, baseline_predictions, zero_division=0
            ),
            "minority recall": recall_score(
                y_test, baseline_predictions, zero_division=0
            ),
            "minority F1": f1_score(
                y_test, baseline_predictions, zero_division=0
            ),
        },
        {
            "model": "class_weight='balanced'",
            "accuracy": accuracy_score(y_test, balanced_predictions),
            "minority precision": precision_score(
                y_test, balanced_predictions, zero_division=0
            ),
            "minority recall": recall_score(
                y_test, balanced_predictions, zero_division=0
            ),
            "minority F1": f1_score(
                y_test, balanced_predictions, zero_division=0
            ),
        },
    ]
).set_index("model").round(3)


,accuracy,minority precision,minority recall,minority F1
model,,,,
unweighted,0.958,1.000,0.231,0.375
class_weight='balanced',0.746,0.147,0.769,0.247


## 4. Oversample training data

Random oversampling duplicates existing minority examples; SMOTE creates synthetic examples by interpolating between nearby minority examples. Both change the training data rather than the class penalties. They require the optional `imbalanced-learn` package (included in the course requirements). Random duplicates may lead to overfitting; synthetic examples may add noise. Try class weights first, and consider SMOTE when there are enough representative minority examples.

Keep scaling and resampling inside an imbalanced-learn pipeline: resampling then happens only on each training fold, never on validation or test rows. SMOTE uses distances, so scale features before applying it. Compare methods on the same untouched test set using minority-class precision, recall, and F1.


In [5]:
from imblearn.over_sampling import RandomOverSampler, SMOTE
from imblearn.pipeline import make_pipeline as make_resampling_pipeline
from sklearn.model_selection import cross_val_score


random_oversampling_model = make_resampling_pipeline(
    StandardScaler(),
    RandomOverSampler(random_state=42),
    LogisticRegression(max_iter=1000),
)
smote_model = make_resampling_pipeline(
    StandardScaler(),
    SMOTE(random_state=42),
    LogisticRegression(max_iter=1000),
)

smote_cv_f1 = cross_val_score(
    smote_model,
    X_train,
    y_train,
    cv=5,
    scoring="f1",
)
print("SMOTE training-fold F1 scores:", smote_cv_f1.round(3))
print("Mean cross-validation F1:", round(smote_cv_f1.mean(), 3))


SMOTE training-fold F1 scores: [0.189 0.185 0.286 0.273 0.145]
Mean cross-validation F1: 0.216


In [6]:
resampling_results = []
for name, resampling_model in {
    "random oversampling": random_oversampling_model,
    "SMOTE": smote_model,
}.items():
    resampling_model.fit(X_train, y_train)
    predictions = resampling_model.predict(X_test)
    resampling_results.append(
        {
            "model": name,
            "minority precision": precision_score(
                y_test, predictions, zero_division=0
            ),
            "minority recall": recall_score(
                y_test, predictions, zero_division=0
            ),
            "minority F1": f1_score(y_test, predictions, zero_division=0),
        }
    )

pd.DataFrame(resampling_results).set_index("model").round(3)


,minority precision,minority recall,minority F1
model,,,
random oversampling,0.147,0.769,0.247
SMOTE,0.139,0.769,0.235


## Key ideas

- Check precision, recall, and F1 for the minority class; accuracy alone can mislead.
- Class weights change the cost of mistakes; oversampling changes the training examples.
- Resample training folds only. Compare methods using the same untouched test set.
